In [0]:
%pip install databricks-vectorsearch openai


In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# Embeddings + Vector Index (Multi-Provider: OpenAI / Azure)

# %pip install databricks-vectorsearch openai sentence-transformers

import os
from databricks.vector_search.client import VectorSearchClient
from openai import OpenAI, AzureOpenAI

# -----------------------------
# CONFIG
# -----------------------------
CATALOG = "vb_rag_demo"
SCHEMA = "rag_demo"

SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_chunks"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.docs_index"
VECTOR_SEARCH_ENDPOINT = "rag-demo-endpoint"

#  SWITCH PROVIDER HERE
PROVIDER = "local"   # "openai" | "azure" | "local"

# -----------------------------
# INIT CLIENTS
# -----------------------------
openai_client = None
azure_client = None
local_model = None

if PROVIDER == "openai":
    openai_client = OpenAI(
        api_key=dbutils.secrets.get("rag-scope", "openai-api-key")
    )
    EMBEDDING_MODEL = "text-embedding-3-small"

elif PROVIDER == "azure":
    azure_client = AzureOpenAI(
        api_key=dbutils.secrets.get("rag-scope", "azure-openai-api-key"),
        api_version="2024-02-01",
        azure_endpoint=dbutils.secrets.get("rag-scope", "azure-openai-endpoint"),
    )
    EMBEDDING_MODEL = dbutils.secrets.get(
        "rag-scope", "azure-openai-embedding-deployment"
    )

elif PROVIDER == "local":
    from sentence_transformers import SentenceTransformer
    local_model = SentenceTransformer("all-MiniLM-L6-v2")
    EMBEDDING_MODEL = "all-MiniLM-L6-v2"

else:
    raise ValueError("Invalid PROVIDER selected")

# -----------------------------
# EMBEDDING FUNCTION (KEY PART)
# -----------------------------
def get_embedding(text):
    try:
        if PROVIDER == "openai":
            resp = openai_client.embeddings.create(
                model=EMBEDDING_MODEL,
                input=text
            )
            return resp.data[0].embedding

        elif PROVIDER == "azure":
            resp = azure_client.embeddings.create(
                model=EMBEDDING_MODEL,
                input=text
            )
            return resp.data[0].embedding

        elif PROVIDER == "local":
            return local_model.encode(text).tolist()

    except Exception as e:
        print(f"Embedding error: {e}")
        return [0.0] * 1536


# -----------------------------
# LOAD DATA
# -----------------------------
pdf = spark.table(SILVER_TABLE).toPandas()

print(f"Generating embeddings for {len(pdf)} chunks...")

# -----------------------------
# GENERATE EMBEDDINGS
# -----------------------------
embeddings = []

for idx, text in enumerate(pdf["chunk_text"].tolist()):
    emb = get_embedding(text)
    embeddings.append(emb)

    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(pdf)}")

pdf["embedding"] = embeddings

# -----------------------------
# SAVE BACK TO TABLE
# -----------------------------
spark.createDataFrame(pdf) \
    .write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print("Embeddings saved to table")

# -----------------------------
# VECTOR SEARCH
# -----------------------------
vsc = VectorSearchClient()

# Create endpoint
try:
    vsc.create_endpoint(name=VECTOR_SEARCH_ENDPOINT, endpoint_type="STANDARD")
    print(f"Endpoint created: {VECTOR_SEARCH_ENDPOINT}")
except Exception:
    print("Endpoint exists")

# Create index
try:
    vsc.create_direct_access_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT,
        index_name=INDEX_NAME,
        primary_key="chunk_id",
        embedding_dimension=len(embeddings[0]) if embeddings else 1536,
        embedding_vector_column="embedding",
        schema={
            "chunk_id": "string",
            "document_id": "string",
            "file_name": "string",
            "chunk_text": "string",
            "chunk_order": "int",
            "source_path": "string",
            "ingested_at": "timestamp",
            "embedding": "array<float>"
        }
    )
    print(f"Index created: {INDEX_NAME}")
except Exception:
    print("Index exists")

# -----------------------------
# UPSERT DATA
# -----------------------------
index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=INDEX_NAME
)

pdf["ingested_at"] = pdf["ingested_at"].apply(lambda x: x.isoformat())

print("DONE: Vector index ready!")

In [0]:
display(spark.table(SILVER_TABLE))

In [0]:
%sql SHOW TABLES IN vb_rag_demo.rag_demo;

In [0]:
try:
    idx = vsc.get_index(
        endpoint_name="rag-demo-endpoint",
        index_name="vb_rag_demo.rag_demo.docs_index"
    )
    print("Index exists")
except:
    print("Index not found")

In [0]:
index = vsc.get_index(
    endpoint_name="rag-demo-endpoint",
    index_name="vb_rag_demo.rag_demo.docs_index"
)

display(index.describe())

In [0]:
%pip install sentence-transformers databricks-vectorsearch

In [0]:
# Databricks notebook source

# Install once
# %pip install sentence-transformers databricks-vectorsearch

from sentence_transformers import SentenceTransformer
from databricks.sdk.service import VectorSearchClient
import pandas as pd

# -----------------------------
# CONFIG
# -----------------------------
CATALOG = "main"
SCHEMA = "rag_demo"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_chunks"

INDEX_NAME = f"{CATALOG}.{SCHEMA}.docs_index_local"
VECTOR_SEARCH_ENDPOINT = "rag-demo-endpoint"

# -----------------------------
# LOAD DATA
# -----------------------------
df = spark.table(SILVER_TABLE).toPandas()

# -----------------------------
# EMBEDDING MODEL (384 DIM)
# -----------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Generating embeddings...")

df["embedding"] = df["chunk_text"].apply(
    lambda x: model.encode(x).tolist()
)

print("Embeddings created")

# -----------------------------
# SAVE BACK TO TABLE
# -----------------------------
spark.createDataFrame(df).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(SILVER_TABLE)

print("Saved embeddings to table")

# -----------------------------
# VECTOR SEARCH
# -----------------------------
vsc = VectorSearchClient()

# Create endpoint (ignore if exists)
try:
    vsc.create_endpoint(
        name=VECTOR_SEARCH_ENDPOINT,
        endpoint_type="STANDARD"
    )
    print("Endpoint created")
except Exception as e:
    print("Endpoint exists:", e)

# Delete old index (important)
try:
    vsc.delete_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT,
        index_name=INDEX_NAME
    )
    print("Old index deleted")
except Exception as e:
    print("No old index:", e)

# Create new index (384 dim)
vsc.create_direct_access_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    primary_key="chunk_id",
    embedding_dimension=384,
    embedding_vector_column="embedding",
    schema={
        "chunk_id": "string",
        "document_id": "string",
        "file_name": "string",
        "chunk_text": "string",
        "chunk_order": "int",
        "source_path": "string",
        "ingested_at": "timestamp",
        "embedding": "array<float>"
    }
)

print("Index created")

# -----------------------------
# UPSERT DATA
# -----------------------------
index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=INDEX_NAME
)

records = spark.table(SILVER_TABLE).toPandas().to_dict(orient="records")

index.upsert(records)

print("✅ Vector index populated successfully!")